# TrojanLens — Colab campaign (reviewer response)

Extends the clean notebook with the 30-fold LOO fine-tune, threshold/IG
sensitivity, and the raw-LLM faithfulness baseline.

**Run order:** Runtime -> Change runtime type -> **T4 GPU**. Run **Cell 1**, then
**Runtime -> Restart session**, then Cell 2 -> end. Upload `combined.jsonl` at
Cell 3. **Cell 9 is long (~2-4 h) and resumable** — if Colab disconnects, just
re-run Cell 9 and it skips finished variants.

In [ ]:
# Cell 1 — install deps, then RESTART SESSION before Cell 2
!pip -q install "transformers>=4.44" "peft>=0.11" accelerate captum scikit-learn pyyaml 2>/dev/null
!pip uninstall -y torchao
print("deps installed -> Runtime > Restart session, then run Cell 2 onward")

In [ ]:
# Cell 2 — GPU check
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
print(torch.cuda.get_device_name(0) if DEVICE=="cuda" else "No GPU: Runtime>Change runtime type>GPU")

In [ ]:
# Cell 3 — upload combined.jsonl
import os, json
if not os.path.exists("combined.jsonl"):
    from google.colab import files
    up = files.upload(); n = list(up.keys())[0]
    if n != "combined.jsonl": os.rename(n, "combined.jsonl")
recs = [json.loads(l) for l in open("combined.jsonl") if l.strip()]
print("loaded", len(recs), "records;", sum(int(r['label']) for r in recs), "positive")

In [ ]:
# Cell 4 — config
import random, numpy as np, torch
CFG = {
    "model_name": "Qwen/Qwen2.5-Coder-0.5B",
    "dtype": torch.bfloat16, "max_seq_len": 1024,
    "lora": dict(r=16, alpha=32, dropout=0.05,
                 target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]),
    "epochs": 1, "lr": 1e-4,
    "focal_gamma": 2.0, "focal_alpha": 0.75, "loc_pos_weight": 8.0,
    "top_k": 15, "tau_g": 0.3, "tau_c": 0.05, "seed": 1234,
}
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
print("config ready:", CFG["model_name"], "| epochs", CFG["epochs"], "| seq", CFG["max_seq_len"])

In [ ]:
# Cell 5 — model (LoRA + heads) + helpers  [proven, unchanged]
import torch.nn as nn, torch.nn.functional as F
from transformers import AutoModel
from peft import LoraConfig, get_peft_model

def focal_loss(logits, targets, gamma, alpha):
    ce = F.cross_entropy(logits, targets, reduction="none"); pt = torch.exp(-ce)
    at = torch.where(targets==1, torch.full_like(ce, alpha), torch.full_like(ce, 1-alpha))
    return (at*(1-pt)**gamma*ce).mean()

def aggregate_to_lines(vals, token_line, reduce="mean"):
    b={}
    for v,ln in zip(vals, token_line): b.setdefault(int(ln),[]).append(float(v))
    return {ln:(sum(x)/len(x) if reduce=="mean" else sum(x)) for ln,x in b.items()}

def neutralize(input_ids, token_line, lines, mask_id=0):
    m=set(int(x) for x in lines)
    return [mask_id if int(token_line[i]) in m else input_ids[i] for i in range(len(input_ids))]

class TrojanLens(nn.Module):
    def __init__(self, cfg, lora=True):
        super().__init__()
        enc = AutoModel.from_pretrained(cfg["model_name"], torch_dtype=cfg["dtype"])
        if lora:
            enc = get_peft_model(enc, LoraConfig(
                r=cfg["lora"]["r"], lora_alpha=cfg["lora"]["alpha"], lora_dropout=cfg["lora"]["dropout"],
                target_modules=cfg["lora"]["target_modules"], bias="none", task_type="FEATURE_EXTRACTION"))
            enc.enable_input_require_grads(); enc.gradient_checkpointing_enable()
        else:
            for p in enc.parameters(): p.requires_grad_(False)
        self.encoder = enc
        self.hidden = getattr(getattr(enc,"config",None),"hidden_size",None) or enc.base_model.config.hidden_size
        self.detect = nn.Linear(self.hidden, 2); self.locate = nn.Linear(self.hidden, 1)
    def embedding_layer(self):
        e=self.encoder
        return e.get_input_embeddings() if hasattr(e,"get_input_embeddings") else e.base_model.get_input_embeddings()
    def encode(self, ids, attn): return self.encoder(input_ids=ids, attention_mask=attn).last_hidden_state
    def forward(self, ids, attn):
        if ids.dim()==1: ids=ids.unsqueeze(0); attn=attn.unsqueeze(0)
        h=self.encode(ids, attn); m=attn.unsqueeze(-1).to(h.dtype)
        pooled=(h*m).sum(1)/m.sum(1).clamp(min=1.0)
        return {"det": self.detect(pooled.float()), "tok": self.locate(h.float()).squeeze(-1)}
    def prob(self, ids, attn):
        with torch.no_grad(): return torch.softmax(self.forward(ids, attn)["det"], -1)[0,1].item()
print("model class ready")

In [ ]:
# Cell 6 — train_model  [proven, unchanged]
def train_model(cfg, train_recs, lora=True, quiet=False):
    net = TrojanLens(cfg, lora=lora).to(DEVICE); net.train()
    params = [p for p in net.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"])
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([cfg["loc_pos_weight"]], device=DEVICE))
    for ep in range(cfg["epochs"]):
        random.shuffle(train_recs); tot=0.0
        for r in train_recs:
            ids = torch.tensor(r["input_ids"][:cfg["max_seq_len"]], dtype=torch.long, device=DEVICE)
            attn = torch.ones_like(ids); out = net(ids, attn)
            y = torch.tensor([int(r["label"])], device=DEVICE)
            ld = focal_loss(out["det"], y, cfg["focal_gamma"], cfg["focal_alpha"])
            tl = r["token_line"][:cfg["max_seq_len"]]; tset=set(int(x) for x in r["trojan_lines"])
            tgt = torch.tensor([1.0 if int(x) in tset else 0.0 for x in tl], device=DEVICE)
            ll = bce(out["tok"][0][:len(tgt)], tgt)
            opt.zero_grad(); (ld+ll).backward(); opt.step(); tot += float(ld+ll)
        if not quiet: print(f"  epoch {ep+1}/{cfg['epochs']}  avg loss {tot/max(1,len(train_recs)):.4f}")
    net.eval(); return net
print("train_model ready")

In [ ]:
# Cell 7 — attribution + verification  [proven; verify accepts tau overrides]
def attribute(net, r, ig_steps=0):
    """grad*input (ig_steps=0) or integrated gradients over the embedding layer."""
    ids = torch.tensor(r["input_ids"][:CFG["max_seq_len"]], dtype=torch.long, device=DEVICE).unsqueeze(0)
    attn = torch.ones_like(ids)
    if ig_steps and ig_steps > 1:
        from captum.attr import LayerIntegratedGradients
        def fwd(i, a): return net(i, a)["det"]
        lig = LayerIntegratedGradients(fwd, net.embedding_layer())
        at = lig.attribute(inputs=ids, baselines=torch.zeros_like(ids),
                           additional_forward_args=(attn,), target=1,
                           n_steps=ig_steps, internal_batch_size=1)
        sal = at.sum(-1)[0].detach().float().cpu().tolist()
    else:
        emb=net.embedding_layer(); cap={}
        def hook(_m,_i,o): o.requires_grad_(True); o.retain_grad(); cap["e"]=o; return o
        hd = emb.register_forward_hook(hook)
        try:
            net.zero_grad(set_to_none=True); out=net(ids, attn); out["det"][0,1].backward()
            sal = (cap["e"]*cap["e"].grad).sum(-1)[0].detach().float().cpu().tolist()
        finally:
            hd.remove()
    ls = aggregate_to_lines([abs(x) for x in sal], r["token_line"][:CFG["max_seq_len"]], "sum")
    return [int(l) for l,_ in sorted(ls.items(), key=lambda kv: kv[1], reverse=True)[:CFG["top_k"]]]

def verify(net, r, cited, topk):
    ids=r["input_ids"][:CFG["max_seq_len"]]; tl=r["token_line"][:CFG["max_seq_len"]]
    def p(seq):
        t=torch.tensor(seq, dtype=torch.long, device=DEVICE); return net.prob(t, torch.ones_like(t))
    base=p(ids); yc=set(cited); AG=len(set(topk)&yc)/max(1,len(yc))
    dc=base-p(neutralize(ids, tl, cited)); flip=(base>=0.5) and ((base-dc)<0.5)
    import random as _r
    others=[l for l in set(int(x) for x in tl) if l not in yc]
    ctrl=_r.sample(others, min(len(cited), len(others))) if others else []
    hold = p(neutralize(ids, tl, ctrl))>=0.5
    return dict(verified=bool(AG>=CFG["tau_g"] and dc>=CFG["tau_c"] and flip and hold),
                AG=AG, delta_c=dc, delta_s=base-p(neutralize(ids, tl, others)), flip=flip)
print("attribution + verification ready")

In [ ]:
# Cell 8 — per-record evaluation helper (returns components)
def eval_record(net, r):
    ids=torch.tensor(r["input_ids"][:CFG["max_seq_len"]], dtype=torch.long, device=DEVICE)
    prob=net.prob(ids, torch.ones_like(ids)); yp=int(prob>=0.5)
    with torch.no_grad():
        tok=net(ids, torch.ones_like(ids))["tok"][0].float().cpu().tolist()
    lp=aggregate_to_lines([torch.sigmoid(torch.tensor(t)).item() for t in tok],
                          r["token_line"][:CFG["max_seq_len"]], "mean")
    pred=[l for l,s in lp.items() if s>=0.5] or [int(l) for l,_ in sorted(lp.items(), key=lambda kv: kv[1], reverse=True)[:CFG["top_k"]]]
    out=dict(file=r["file"], y_true=int(r["label"]), y_pred=yp,
             gt=sorted(int(x) for x in r["trojan_lines"]), pred=sorted(int(x) for x in pred), verified=False)
    if yp==1 or int(r["label"])==1:
        topk=attribute(net, r); v=verify(net, r, pred[:CFG["top_k"]], topk)
        out.update(verified=v["verified"], AG=v["AG"], delta_c=v["delta_c"], delta_s=v["delta_s"], flip=v["flip"], topk=topk)
    return out

def variant_of(f): return f.replace("\\","/").split("/")[0]
def agg_metrics(rows):
    tp=fp=fn=tn=0
    for p in rows:
        if p["y_true"] and p["y_pred"]: tp+=1
        elif p["y_pred"] and not p["y_true"]: fp+=1
        elif p["y_true"] and not p["y_pred"]: fn+=1
        else: tn+=1
    prec=tp/(tp+fp) if tp+fp else 0; rec=tp/(tp+fn) if tp+fn else 0
    f1=2*prec*rec/(prec+rec) if prec+rec else 0
    pos=[p for p in rows if p["y_true"]]
    plc=sum(1 for p in pos if p["gt"] and len(set(p["gt"])&set(p["pred"]))/len(p["gt"])>=0.5)/max(1,len(pos))
    iou=sum((len(set(p["gt"])&set(p["pred"]))/len(set(p["gt"])|set(p["pred"]))) if (p["gt"] or p["pred"]) else 0 for p in pos)/max(1,len(pos))
    pp=[p for p in rows if p["y_pred"]]; vr=sum(1 for p in pp if p.get("verified"))/max(1,len(pp))
    return dict(f1=round(f1,3), precision=round(prec,3), recall=round(rec,3),
                PLC=round(plc,3), IoU=round(iou,3), VR=round(vr,3), n=len(rows),
                tp=tp, fp=fp, fn=fn, tn=tn)
print("eval helpers ready")

In [ ]:
# Cell 9 — 30-fold LOO FINE-TUNE campaign (RESUMABLE; long: ~2-4 h on T4)
import gc, os, json
OUT="loo_ft_records.jsonl"
done=set()
if os.path.exists(OUT):
    for l in open(OUT):
        try: done.add(json.loads(l)["held"])
        except Exception: pass
pos_vars=sorted(set(variant_of(r["file"]) for r in recs if int(r["label"])==1))
print(f"{len(pos_vars)} trojan variants; {len(done)} already finished")
f=open(OUT,"a")
for i,held in enumerate(pos_vars):
    if held in done:
        print(f"[skip] {held}"); continue
    tr=[r for r in recs if variant_of(r["file"])!=held]
    te=[r for r in recs if variant_of(r["file"])==held]
    net=train_model(CFG, tr, lora=True, quiet=True)
    for r in te:
        row=eval_record(net, r); row["held"]=held; row["family"]=held.split("-")[0]
        f.write(json.dumps(row)+"\n"); f.flush()
    del net; gc.collect(); torch.cuda.empty_cache()
    print(f"[{i+1}/{len(pos_vars)}] {held} done")
f.close(); print("LOO fine-tune campaign complete ->", OUT)

In [ ]:
# Cell 10 — aggregate LOO fine-tune (overall + per family)
import json
rows=[json.loads(l) for l in open("loo_ft_records.jsonl") if l.strip()]
print("total evaluated:", len(rows))
overall=agg_metrics(rows); print("\nFINE-TUNED (LOO) overall:", overall)
for fam in sorted(set(r["family"] for r in rows)):
    fr=[r for r in rows if r["family"]==fam]
    print(f"  {fam:<7}:", agg_metrics(fr))
print("\n>>> Table 6 fine-tuned-LOO row: F1", overall["f1"], "IoU", overall["IoU"], "PLC", overall["PLC"])
print(">>> Table 7 fine-tuned-LOO row: VR", overall["VR"])

In [ ]:
# Cell 11 — threshold sweep: VR vs (tau_g, tau_c) from dumped components [M4]
import json
rows=[json.loads(l) for l in open("loo_ft_records.jsonl") if l.strip()]
pos_pred=[r for r in rows if r["y_pred"]==1 and "AG" in r]  # positive predictions with components
def vr_at(tg, tc):
    if not pos_pred: return 0.0
    return round(sum(1 for r in pos_pred if r["AG"]>=tg and r["delta_c"]>=tc and r.get("flip"))/len(pos_pred),3)
tgs=[0.2,0.3,0.4]; tcs=[0.03,0.05,0.10]
print("VR sensitivity (rows=tau_g, cols=tau_c):")
print("        " + "  ".join(f"tc={t}" for t in tcs))
for tg in tgs:
    print(f"tg={tg}  " + "   ".join(f"{vr_at(tg,tc):.3f}" for tc in tcs))
print("\n(default used in paper: tau_g=0.3, tau_c=0.05)")

In [ ]:
# Cell 12 — IG-step sensitivity: grad*input vs IG T in {32,64,128} [m8]
import json, random
rows=[json.loads(l) for l in open("loo_ft_records.jsonl") if l.strip()]
# retrain one representative fold to have a live model, then compare AG/VR vs attribution
sample_held = sorted(set(r["held"] for r in rows if r["family"]=="AES"))[0]
tr=[r for r in recs if variant_of(r["file"])!=sample_held]
te=[r for r in recs if variant_of(r["file"])==sample_held and int(r["label"])==1]
net=train_model(CFG, tr, lora=True, quiet=True)
for steps in [0,32,64,128]:
    ags=[]; ver=0
    for r in te:
        e=eval_record(net, r)
        topk=attribute(net, r, ig_steps=steps)
        v=verify(net, r, e["pred"][:CFG["top_k"]], topk)
        ags.append(v["AG"]); ver+=int(v["verified"])
    lbl="grad*input" if steps==0 else f"IG T={steps}"
    print(f"{lbl:<12}: mean AG {sum(ags)/max(1,len(ags)):.3f} | verified {ver}/{len(te)}")
import gc; del net; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 13 — raw-LLM rationale faithfulness -> Table 7 'Raw-LLM' row
import re, gc, json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
# a fine-tuned detector is needed for the counterfactual checks; retrain one fold
held=sorted(set(variant_of(r["file"]) for r in recs if int(r["label"])==1))[0]
tr=[r for r in recs if variant_of(r["file"])!=held]
te=[r for r in recs if variant_of(r["file"])==held and int(r["label"])==1]
det=train_model(CFG, tr, lora=True, quiet=True)
ZS="Qwen/Qwen2.5-Coder-0.5B-Instruct"
zt=AutoTokenizer.from_pretrained(ZS)
zl=AutoModelForCausalLM.from_pretrained(ZS, torch_dtype=torch.bfloat16).to(DEVICE).eval()
def raw_llm_cited(r, mx=120):
    numbered="\n".join(f"{i+1}: {ln}" for i,ln in enumerate(r["lines"][:mx]))
    pr=("You are a hardware-security auditor. The Verilog below is numbered by line. "
        "List ONLY the line numbers that implement the hardware Trojan (trigger or "
        "payload), comma-separated. If unsure, give your best guess.\n\n"+numbered+"\n\nTrojan lines:")
    text=zt.apply_chat_template([{"role":"user","content":pr}], tokenize=False, add_generation_prompt=True)
    enc=zt(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad(): out=zl.generate(**enc, max_new_tokens=40, do_sample=False)
    ans=zt.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    nums=[int(x) for x in re.findall(r"\d+", ans)][:CFG["top_k"]]
    return sorted(set(n for n in nums if 1<=n<=len(r["lines"])))
rawrows=[]
for r in te:
    cited=raw_llm_cited(r)
    if not cited: cited=[1]
    topk=attribute(det, r); v=verify(det, r, cited, topk)
    rawrows.append(dict(y_true=1, y_pred=1, gt=sorted(int(x) for x in r["trojan_lines"]),
                        pred=cited, verified=v["verified"], AG=v["AG"], delta_c=v["delta_c"], delta_s=v["delta_s"], flip=v["flip"]))
mr=agg_metrics(rawrows)
mean=lambda a:round(sum(a)/len(a),3) if a else 0.0
print(">>> Table 7 Raw-LLM row: delta_c", mean([x["delta_c"] for x in rawrows]),
      "delta_s", mean([x["delta_s"] for x in rawrows]),
      "AG", mean([x["AG"] for x in rawrows]), "VR", mr["VR"])
del det, zl; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 14 — frozen-probe LOO (VR under LOO, no fine-tune) for the ablation
import gc, os, json
OUT="loo_probe_records.jsonl"
done=set()
if os.path.exists(OUT):
    for l in open(OUT):
        try: done.add(json.loads(l)["held"])
        except Exception: pass
pos_vars=sorted(set(variant_of(r["file"]) for r in recs if int(r["label"])==1))
f=open(OUT,"a")
for i,held in enumerate(pos_vars):
    if held in done: continue
    tr=[r for r in recs if variant_of(r["file"])!=held]
    te=[r for r in recs if variant_of(r["file"])==held]
    net=train_model(CFG, tr, lora=False, quiet=True)   # heads only, frozen backbone
    for r in te:
        row=eval_record(net, r); row["held"]=held; row["family"]=held.split("-")[0]
        f.write(json.dumps(row)+"\n"); f.flush()
    del net; gc.collect(); torch.cuda.empty_cache()
    print(f"[{i+1}/{len(pos_vars)}] probe {held} done")
f.close()
rows=[json.loads(l) for l in open(OUT) if l.strip()]
print("\nFROZEN PROBE (LOO):", agg_metrics(rows))

## Cell 15 — GNN baselines for Table 6 (reported numbers + caveat)

No public code is available for GNN-MFF, and GNN4TJ / TrojanLoC train and test on
different splits and datasets than ours, so we cite their **reported** figures
with an explicit different-split caveat rather than claim a same-split
re-implementation:

| Method | Reported Det. F1 | Note |
|---|---|---|
| GNN4TJ (Yasaei et al., DATE'21) | ~0.92 | RTL dataflow graphs; different split |
| GNN-MFF (Zhang et al., 2025) | 0.9708 | multi-view graph; extended Trust-Hub; no public code |
| TrojanLoC (Xiao et al., 2025) | 0.99 det / 0.92-0.93 line macro-F1 | synthetic TrojanInS data |

If you later want a same-split GNN4TJ, its repo builds an RTL dataflow graph per
design; that graph-construction step is the heavy part and is best run as its own
job rather than in this notebook.